In [2]:
%cd nanoVLM

import gymnasium as gym
from minigrid.envs.empty import EmptyEnv
import numpy as np
import torch
from PIL import Image

from data.processors import get_image_processor, get_image_string
from data.emptyenv_action_dataset import DEFAULT_PROMPT

from models.vision_language_model_action import VisionLanguageActionModel
import models.config as config
from data.emptyenv_action_dataset import DEFAULT_PROMPT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# загружаем конфиг
vlm_cfg = config.VLMConfig()
# загружаем модель
#model = VisionLanguageActionModel(vlm_cfg, load_backbone=True, num_actions=3)
vlm_cfg.max_img_size = 512
model = VisionLanguageActionModel.from_pretrained("/teamspace/studios/this_studio/nanoVLM/checkpoints_emptyenv_action/step_500_1.0")

model.to(device)
model.eval()

print("Model loaded.")



/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


/teamspace/studios/this_studio/nanoVLM


Model loaded.


In [37]:
import torch
import numpy as np
from PIL import Image
from minigrid.envs.empty import EmptyEnv
from data.processors import get_image_processor, get_image_string
from data.emptyenv_action_dataset import DEFAULT_PROMPT
import matplotlib.pyplot as plt

ID2ACTION = {0: "left", 1: "right", 2: "forward"}
ACTION2ID = {"left": 0, "right": 1, "forward": 2}

device = next(model.parameters()).device

image_processor = get_image_processor(
    vlm_cfg.max_img_size,
    vlm_cfg.vit_img_size,
    vlm_cfg.resize_to_max_side_len
)
tokenizer = model.tokenizer


@torch.no_grad()
def policy_action_from_rgb(rgb, prompt=DEFAULT_PROMPT, temperature=1.0, greedy=False):
    """
    temperature > 1.0  -> more exploration (flatter probs)
    temperature < 1.0  -> more greedy (sharper probs)
    greedy=True        -> argmax regardless of temperature
    """
    img = Image.fromarray(rgb).convert("RGB")
    processed_image, splitted_image_count = image_processor(img)

    messages = [{"role": "user", "content": prompt}]
    image_string = get_image_string(tokenizer, [splitted_image_count], vlm_cfg.mp_image_token_length)
    messages[0]["content"] = image_string + messages[0]["content"]

    conv = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_special_tokens=False,
        return_dict=True,
    )

    input_ids = torch.tensor(conv["input_ids"], dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.tensor(conv["attention_mask"], dtype=torch.long).unsqueeze(0).to(device)
    images = [[processed_image]]  # batch size 1

    logits, _ = model(
        input_ids=input_ids,
        images=images,
        attention_mask=attention_mask,
        action_labels=None
    )  # logits: [1, 3]

    logits = logits[0]  # [3]

    if greedy:
        probs = torch.softmax(logits, dim=-1)
        action = int(torch.argmax(probs).item())
        return action, probs.detach().cpu().numpy()

    # Temperature sampling
    T = float(temperature)
    if T <= 0:
        raise ValueError("temperature must be > 0")

    probs = torch.softmax(logits / T, dim=-1)   # [3]
    action = int(torch.multinomial(probs, num_samples=1).item())
    return action, probs.detach().cpu().numpy()


def run_eval(num_episodes=20, sizes=[6, 7], max_steps=200, render=False, seed=0, temperature=1.0, greedy=False):
    successes = 0
    returns = []
    lengths = []

    rng = np.random.default_rng(seed)

    for ep in range(num_episodes):
        size = int(rng.choice(sizes))
        env = EmptyEnv(size=size, agent_start_pos=None, render_mode="rgb_array")
        obs, info = env.reset(seed=seed + ep)

        env.agent_dir = int(rng.integers(0, 4))

        ep_ret = 0.0
        ep_len = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            rgb = env.render()

            action, probs = policy_action_from_rgb(
                rgb,
                temperature=temperature,
                greedy=greedy
            )

            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += float(reward)
            ep_len += 1

            if render:
                print(f"ep={ep} t={ep_len} a={ID2ACTION[action]} probs={np.round(probs,3)}")
                plt.figure(figsize=(4,4))
                plt.imshow(rgb)
                plt.axis("off")
                plt.show()

            if ep_len >= max_steps:
                break

        success = 1 if ep_ret > 0 else 0
        successes += success
        returns.append(ep_ret)
        lengths.append(ep_len)

        print(f"[EP {ep}] size={size} success={success} return={ep_ret:.2f} len={ep_len}")

        env.close()

    print("\n=== Summary ===")
    print(f"Success rate: {successes/num_episodes:.3f} ({successes}/{num_episodes})")
    print(f"Avg return:   {np.mean(returns):.3f}")
    print(f"Avg length:   {np.mean(lengths):.1f}")



# 1) exploration (temperature sampling)
run_eval(num_episodes=100, sizes=[8], max_steps=100, render=False, seed=123, temperature=1.7, greedy=False) 

# run_eval(num_episodes=20, sizes=[8], max_steps=100, render=True, seed=123, greedy=True)

Resize to max side len: True


[EP 0] size=8 success=0 return=0.00 len=100
[EP 1] size=8 success=1 return=0.99 len=2
[EP 2] size=8 success=0 return=0.00 len=100
[EP 3] size=8 success=0 return=0.00 len=100
[EP 4] size=8 success=0 return=0.00 len=100
[EP 5] size=8 success=1 return=0.91 len=27
[EP 6] size=8 success=0 return=0.00 len=100
[EP 7] size=8 success=0 return=0.00 len=100
[EP 8] size=8 success=1 return=0.94 len=17
[EP 9] size=8 success=1 return=0.90 len=28
[EP 10] size=8 success=0 return=0.00 len=100
[EP 11] size=8 success=1 return=0.90 len=29
[EP 12] size=8 success=0 return=0.00 len=100
[EP 13] size=8 success=0 return=0.00 len=100
[EP 14] size=8 success=1 return=0.99 len=4
[EP 15] size=8 success=1 return=0.83 len=47
[EP 16] size=8 success=0 return=0.00 len=100
[EP 17] size=8 success=1 return=0.85 len=43
[EP 18] size=8 success=1 return=0.94 len=17
[EP 19] size=8 success=1 return=0.96 len=10
[EP 20] size=8 success=1 return=0.84 len=46
[EP 21] size=8 success=1 return=0.79 len=60
[EP 22] size=8 success=1 return=0.

In [38]:
run_eval(num_episodes=100, sizes=[8], max_steps=100, render=False, seed=123, temperature=1.8, greedy=False) 


[EP 0] size=8 success=0 return=0.00 len=100
[EP 1] size=8 success=1 return=0.97 len=8
[EP 2] size=8 success=1 return=0.83 len=47
[EP 3] size=8 success=0 return=0.00 len=100
[EP 4] size=8 success=1 return=0.94 len=17
[EP 5] size=8 success=1 return=0.93 len=19
[EP 6] size=8 success=0 return=0.00 len=100
[EP 7] size=8 success=0 return=0.00 len=100
[EP 8] size=8 success=1 return=0.96 len=12
[EP 9] size=8 success=1 return=0.77 len=65
[EP 10] size=8 success=0 return=0.00 len=100
[EP 11] size=8 success=1 return=0.75 len=70
[EP 12] size=8 success=1 return=0.95 len=15
[EP 13] size=8 success=0 return=0.00 len=100
[EP 14] size=8 success=1 return=0.99 len=4
[EP 15] size=8 success=1 return=0.96 len=10
[EP 16] size=8 success=0 return=0.00 len=100
[EP 17] size=8 success=1 return=0.67 len=94
[EP 18] size=8 success=1 return=0.94 len=17
[EP 19] size=8 success=1 return=0.92 len=23
[EP 20] size=8 success=1 return=0.97 len=8
[EP 21] size=8 success=0 return=0.00 len=100
[EP 22] size=8 success=1 return=0.88 